In [15]:
!rm -rf logs/ # clear logs
!rm -rf optimizer_output/

# Imports

In [2]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice again
!ngspice -v

PDK_ROOT: /foss/pdks
SPICE_USERINIT_DIR: /foss/pdks/ihp-sg13g2/libs.tech/ngspice
******
** ngspice-44.2 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2024, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Sat May 24 09:38:33 UTC 2025
******


In [3]:
import sympy as sp
import logging

from pathlib import Path

from symxplorer.spice_engine             import Spicelib_Wrapper, Sim_Execution_Type
from symxplorer.designer_tools           import Nevergrad_Spice_Multi_Spec_Optimizer, Project_Setup

from symxplorer.logging import setup_loggers

logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

2025-10-01 07:06:04,850 - SymXplorer.optimizer - Using device: cpu and dtype: torch.float64
2025-10-01 07:06:05,145 - SymXplorer.jupyter - Spicelib_Wrapper imported successfully.


# Instantiations


In [4]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

07:06:08 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
07:06:08 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/logs/SymXplorer_2025-10-01_07-06-08.log
07:06:08 - SymXplorer: [INFO] 🔧 spicelib logger set to 50


In [5]:
# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

07:06:09 - SymXplorer.domains: [INFO] 📂 Loading project setup from /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml
07:06:09 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: CMA, type=nevergrad, budget=200, random_seed=48
07:06:09 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
07:06:09 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
07:06:09 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
07:06:09 - SymXplorer.domains: [INFO] 	Number of target specs: 1
07:06:09 - SymXplorer.domains: [INFO] 		- TargetSpec(name=gain_db, target=10, tolerance=1, goal=OptimizationGoalType.EXCEED, sim_type=SimType.AC, enable=True)
07:06:09 - SymXplorer.domains: [INFO] Project 'Tunable-TIA' initialized with simulator 'ngspice'
07:06:09 - SymXplorer.domains: [INFO] 	Workspace root: /foss/designs/eda/SymXplorer
07:06:09 - SymXplorer.domai

Project_Setup(name='Tunable-TIA', description='Tunable TIA BPF example sizing in the ihp-sg13g2 technology', simulator='ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer'), netlist=PosixPath('examples/tunable-tia/ihp-sg13g2/spice/tb_ac.spice'), outdir=PosixPath('examples/tunable-tia/scripts/optimizer_output'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07), 'max_cap_w': np.float64(0.001), 'min_cap_w': np.float64(1e-06), 'max_cap_l': np.float64(0.001), 'min_cap_l': np.float64(1e-06), 'max_res_w': np.float64(0.001), 'min_res_w': np.float64(1e-06), 'max_res_l': np.float64(0.001), 'min_res_l': np.float64(1e-06)}), pvt=PVT(temp=25, corner='tt', suppl

In [6]:
# (2) Create the Spice Simulator Wrapper
wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.name,
    netlist_filename= PROJECT_SETUP.ws_root / PROJECT_SETUP.netlist,
    output_folder=PROJECT_SETUP.ws_root / PROJECT_SETUP.outdir,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

07:06:09 - SymXplorer.spicelib: [INFO] 📂 Creating output directory for the first time: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
07:06:09 - SymXplorer.spicelib: [INFO] --------------------------------------------------
07:06:09 - SymXplorer.spicelib: [INFO] 🚀 Spicelib_Wrapper initialized successfully!
07:06:09 - SymXplorer.spicelib: [INFO] 	📝 Project: Tunable-TIA
07:06:09 - SymXplorer.spicelib: [INFO] 	📜 Schematic: tb_ac
07:06:09 - SymXplorer.spicelib: [INFO] 	📂 Output Folder: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
07:06:09 - SymXplorer.spicelib: [INFO] --------------------------------------------------
07:06:09 - SymXplorer.spicelib: [INFO] Using ngspice from ['/foss/tools/bin/ngspice']
07:06:09 - SymXplorer.spicelib: [INFO] 📊 --- Circuit Information ---
07:06:09 - SymXplorer.spicelib: [INFO] 🔗 Nodes in the netlist: ['VSS', 'GND', 'VDD', 'Vbias', 'Von', 'Vop', 'In', 'Ip']
07:06:09 - SymXplorer.spicelib: [INFO] Testbe

In [7]:
circuit_optimizer = Nevergrad_Spice_Multi_Spec_Optimizer(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

07:06:10 - SymXplorer.optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 1 target specs


## Sanity Check

In [8]:
wrapper.run_sanity_check(
    use_editor=True,
    sim_execution_t=Sim_Execution_Type.RUN_NOW
)

07:06:11 - SymXplorer.spicelib: [INFO] 📂 Creating dedicated sanity check folder...
07:06:11 - SymXplorer.spicelib: [INFO] 🧪 Running sanity check simulation...
07:06:15 - SymXplorer.spicelib: [INFO] simulator log: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/sanity_check/Tunable-TIA_sanity.log
07:06:15 - SymXplorer.spicelib: [INFO] simulator RAW: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/sanity_check/Tunable-TIA_sanity.raw
07:06:15 - SymXplorer.spicelib: [INFO] 🔎 Verifying simulation results...
07:06:15 - SymXplorer.spicelib: [INFO] ✅ Sanity check passed 🎉


True

# Method Calls

In [9]:
circuit_optimizer.parameterize()

Dict(x_dut_cap_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_cap_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_3_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_3_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}]):{'x_dut_nfet_w': 50.0, 'x_dut_nfet_l': 50.0, 'x_dut_cap_w': 50.0, 'x_dut_cap_l': 50.0, 'x_dut_res_s_l': 50.0, 'x_dut_res_s_w': 50.0, 'x_dut_res_3_l': 50.0, 'x_dut_res_3_w': 50.0}

In [10]:
circuit_optimizer.optimize()

07:06:38 - SymXplorer.optimizer: [INFO] Optimization process started.
07:06:38 - SymXplorer.optimizer: [INFO] Optimizer is set to CMA with budget = 200
Optimizing:  16%|█▌        | 31/200 [00:46<04:10,  1.48s/trial]


KeyboardInterrupt: 

In [11]:
circuit_optimizer.plot_loss(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

07:07:41 - SymXplorer.optimizer: [INFO] 📊 Plot saved to /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/loss_curve.html
07:07:41 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


In [12]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss, metadata = out
metadata

07:07:49 - SymXplorer.optimizer: [INFO] best loss: 0.0


{'gain_db': {'curr_val': np.float64(9.48879532396682),
  'loss': np.float64(0.0)}}

In [13]:
# Print parameter sizes (convert to u)
for param in best_param:
    print(f"{param}: {best_param[param]*1e6 :0.2f}")

x_dut_nfet_w: 5.96
x_dut_nfet_l: 2.82
x_dut_cap_w: 515.58
x_dut_cap_l: 374.37
x_dut_res_s_l: 251.28
x_dut_res_s_w: 155.27
x_dut_res_3_l: 935.85
x_dut_res_3_w: 544.33


In [14]:
circuit_optimizer.plot_solution(best_param, show_plot=True, trace_name="vout")

07:07:57 - SymXplorer.optimizer: [INFO] total loss: 0.0
07:07:57 - SymXplorer.optimizer: [INFO] 	Spec 'gain_db': curr_val=9.48879532396682, loss=0.0


# Testing

In [ ]:
PROJECT_SETUP.optimizer_config.target_specs.list_target_names()

In [ ]:
target_spec = PROJECT_SETUP.optimizer_config.target_specs.get_target_by_name('gain_db')
target_spec

In [ ]:
circuit_optimizer.compute_spec_loss(spec_curr_val=-90, target_spec=target_spec)

In [ ]:
PROJECT_SETUP.dut_params